# Analysis

## Setting parameters

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [59]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass', 
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     # 'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    # 'ecoli',
    # 'wisconsin',
    # 'dermatology',
    # 'cleveland',
    # 'spectfheart',
    # 'bupa',
    # 'yeast',
    # 'heart',
    # 'australian',
    # 'glass',
    # 'pima',
    'australian',
     'bands',
     'bupa',
     'cleveland',
     # 'dermatology', temp 01/07/24
     'ecoli',
     'glass',   # todo tijdelijk bij prior-incl
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features! eventjes skip voor QuickReduct 25/04/2024
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # temp uit voor inclusion tests
     'wisconsin',
     'yeast'  # todo tijdelijk bij prior-incl
]

qr_list = [
    'australian',
    'bupa',
    'cleveland',
    'dermatology',
    'ecoli',
    'glass',
    'heart',
    'pima',
    'spectfheart',
    # 'vehicle',
    # 'vowel',
    'wisconsin',
    'yeast'
]

In [60]:
data_sets = inclusion_test

In [61]:
len(data_sets)

15

In [62]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [63]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [64]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [65]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [66]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [67]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [68]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [69]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [133]:
frri_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    # 'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering-bug' / '1-1e-6',
    # 'cv-thres': Path.cwd() / results_base / 'lower-approx' / 'cv-inclusion',
    'frfs': Path.cwd() / results_base / 'lower-approx' / 'frfs',
    'msecvx-no-reducts': Path.cwd() / results_base / 'multiclassMSECVX' / 'no-reducts',
    'maecvx-no-reducts': Path.cwd() / results_base / 'multiclassMAECVX' / 'no-reducts',
    'msecvx-0.5' : Path.cwd() / 'results' / 'multiclassMSECVX' / 'no-reducts-0.5-threshold',
    'maecvx-0.5-reducts' : Path.cwd() / 'results' / 'multiclassMAECVX' / 'reducts-0.5-threshold',
    'msecvx-0.5-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-0.5-threshold',
    'msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl95-cov05',
    'msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl5-cov05',
    'relabel-msecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-default',
    'relabel-maecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-default',
    'relabel-msecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-cov50',
    'relabel-maecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50',
    'relabel-msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in50',
    'relabel-maecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in50',
    'relabel-msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95',
    'relabel-maecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in95',
    'relabel-msecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-incl5-cov05-v2',
    'relabel-maecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc50',
    'relabel-msecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95-cov50',
    'relabel-maecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc95',
    'base-no-reducts': Path.cwd() / results_base / 'inclusion' / 'no-reducts',
    'owa-lower-base': Path.cwd() / results_base / 'owa-lower' / 'default',
    'default-check': Path.cwd() / 'results' / 'default',
    'prio 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6',
    'prio-max 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scale',
    '1e-6-high' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high',
    '1e-6-high-sum' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high-sum',
    '1e-6-trueprior-influence-1e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-2',
    '1e-6-trueprior-influence-1e-3' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-3',
    # '1e-6-trueprior-influence-1e-4' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-4',
    # '1e-6-trueprior-influence-1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-6',
    '1e-6-scaled_prior-influence-1e-2' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-1e-2',
    '1e-6-scaled_prior-influence-3e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-3e-2',

}

In [134]:
for model, path in frri_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

incl 1e-6
incl 1e-4
incl 1e-3
incl 1e-2
owa 1e-6
frfs
msecvx-no-reducts
maecvx-no-reducts
msecvx-0.5
maecvx-0.5-reducts
msecvx-0.5-reducts
msecvx-i95-reducts
msecvx-i50-reducts
relabel-msecvx-reducts-default
relabel-maecvx-reducts-default
relabel-msecvx-c50-reducts
relabel-maecvx-c50-reducts
relabel-msecvx-i50-reducts
relabel-maecvx-i50-reducts
relabel-msecvx-i95-reducts
relabel-maecvx-i95-reducts
relabel-msecvx-i50+c50-reducts
relabel-maecvx-i50+c50-reducts
relabel-msecvx-i95+c50-reducts
relabel-maecvx-i95+c50-reducts
base-no-reducts
owa-lower-base
default-check
prio 1e-6
prio-max 1e-6
1e-6-high
1e-6-high-sum
1e-6-trueprior-influence-1e-2
1e-6-trueprior-influence-1e-3
1e-6-scaled_prior-influence-1e-2
1e-6-scaled_prior-influence-3e-2


|## Checking all of the models

In [135]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-6', 'incl 1e-4', 'incl 1e-3', 'incl 1e-2', 'owa 1e-6', 'frfs', 'msecvx-no-reducts', 'maecvx-no-reducts', 'msecvx-0.5', 'maecvx-0.5-reducts', 'msecvx-0.5-reducts', 'msecvx-i95-reducts', 'msecvx-i05-reducts', 'relabel-msecvx-i05-reducts', 'base-no-reducts', 'owa-lower-base', 'default-check', 'prio 1e-6', 'prio-max 1e-6', '1e-6-high', '1e-6-high-sum', '1e-6-trueprior-influence-1e-2', '1e-6-trueprior-influence-1e-3', '1e-6-scaled_prior-influence-1e-2', '1e-6-scaled_prior-influence-3e-2', 'relabel-msecvx-reducts-default', 'relabel-msecvx-c05-reducts', 'relabel-msecvx-i05+c05-reducts', 'msecvx-i50-reducts', 'relabel-maecvx-reducts-default', 'relabel-msecvx-c50-reducts', 'relabel-msecvx-i50-reducts', 'relabel-msecvx-i95-reducts', 'relabel-msecvx-i50+c50-reducts', 'relabel-maecvx-c50-reducts', 'relabel-maecvx-i50-reducts', 'relabel-maecvx-i95-reducts', 'relabel-maecvx-i50+c50-reducts', 'relabel-msecvx-

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [136]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        'incl 1e-4',
        # 'incl 1e-3', 
        # 'incl 1e-2', 
        # 'incl 0.95',
        # 'owa 1e-6',
        # 'quickreduct 1e-6',
        'cv-thres',
        # 'frfs',
        # 'prio 1e-6',
        # 'prio-max 1e-6',
        '1e-6-high',
        # '1e-6-high-sum',
        # '1e-6-trueprior-influence-1e-2',
        # '1e-6-trueprior-influence-1e-4',
        '1e-6-scaled_prior-influence-1e-2',
        '1e-6-scaled_prior-influence-3e-2',
    ],
    'quickreduct' : [
        'incl 1e-6',
        'quickreduct 1e-6',
        'frfs'
    ],
    'gran' : [
        'base-no-reducts',
        'msecvx-no-reducts',
        'maecvx-no-reducts',
        'msecvx-0.5',
    ],
    'gran-reducts': [
        'incl 1e-4',
        # 'msecvx-0.5-reducts',
        # 'maecvx-0.5-reducts',
        # 'msecvx-i95-reducts',
        'msecvx-i50-reducts',
        'relabel-maecvx-reducts-default',
        'relabel-maecvx-c50-reducts',
        'relabel-maecvx-i50-reducts',
        'relabel-maecvx-i95-reducts',
        'relabel-maecvx-i50+c50-reducts',
        'relabel-maecvx-i95+c50-reducts',
        'relabel-msecvx-reducts-default',
        'relabel-msecvx-c50-reducts',
        'relabel-msecvx-i50-reducts',
        'relabel-msecvx-i95-reducts',
        'relabel-msecvx-i50+c50-reducts',
        'relabel-msecvx-i95+c50-reducts',

    ],
    'owa' : [
        'incl 1e-4',
        'owa-lower-base'
    ],
    'default-check': [ # -> OK
        'incl 1e-4',
        'default-check'
    ]
}

In [145]:
nr = 'gran-reducts' # 6
save = True

In [146]:
main_folder = nr + '-relabel'  # + "small"  # 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [139]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [140]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]
# table_acc['max'] = table_acc.max(axis='columns', skipna=True)
table_acc.drop(['glass', 'yeast'], inplace=True)
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)
table_acc

,incl 1e-4,msecvx-i50-reducts,relabel-maecvx-reducts-default,relabel-maecvx-c50-reducts,relabel-maecvx-i50-reducts,relabel-maecvx-i95-reducts,relabel-maecvx-i50+c50-reducts,relabel-maecvx-i95+c50-reducts,relabel-msecvx-reducts-default,relabel-msecvx-c50-reducts,relabel-msecvx-i50-reducts,relabel-msecvx-i95-reducts,relabel-msecvx-i50+c50-reducts,relabel-msecvx-i95+c50-reducts
australian,0.787517,0.842375,0.850373,0.850373,0.569359,0.859477,0.86088,0.850373,0.840681,0.844865,0.502179,0.800638,0.856478,0.847267
bands,0.675141,0.6741,0.523886,0.523886,0.5,0.5,0.523886,0.523886,0.530437,0.542326,0.5,0.530437,0.662985,0.542326
bupa,0.600724,0.526257,0.4975,0.5,0.5,0.4975,0.5,0.5,0.495208,0.519359,0.503846,0.495208,0.532221,0.519359
cleveland,0.290397,0.300831,0.312973,0.266758,0.237525,0.258694,0.231389,0.265403,0.291022,0.269662,0.212064,0.276536,0.28676,0.277877
ecoli,0.720659,0.695693,0.247619,0.529192,0.17381,0.247619,0.502696,0.529192,0.247619,0.714563,0.17381,0.247619,0.705842,0.714563
heart,0.7675,0.725,0.780833,0.780833,0.581667,0.7675,0.618333,0.780833,0.743333,0.779167,0.5,0.759167,0.739167,0.786667
ionosphere,0.912468,0.893239,0.671154,0.671154,0.5,0.651282,0.790355,0.671154,0.633974,0.798077,0.514312,0.633974,0.881783,0.798077
pima,0.687633,0.634845,0.5,0.5,0.5,0.5,0.5,0.5,0.526771,0.543993,0.5,0.526771,0.59648,0.543993
sonar,0.786212,0.744394,0.5,0.567222,0.5,0.5,0.724167,0.567222,0.492929,0.786086,0.5,0.492929,0.744394,0.786086
spectfheart,0.628874,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.490584,0.5,0.497727,0.48355,0.5,0.5


In [147]:
if save:
    bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_51409/22476893.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [142]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
# table_rule.drop(['glass', 'yeast'], inplace=True)
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)
table_rule

,incl 1e-4,msecvx-i50-reducts,relabel-maecvx-reducts-default,relabel-maecvx-c50-reducts,relabel-maecvx-i50-reducts,relabel-maecvx-i95-reducts,relabel-maecvx-i50+c50-reducts,relabel-maecvx-i95+c50-reducts,relabel-msecvx-reducts-default,relabel-msecvx-c50-reducts,relabel-msecvx-i50-reducts,relabel-msecvx-i95-reducts,relabel-msecvx-i50+c50-reducts,relabel-msecvx-i95+c50-reducts
australian,92.3,80.6,24.3,24.3,1.5,5.0,9.9,23.9,15.3,139.1,1.6,6.6,48.9,119.4
bands,59.0,84.7,7.5,7.5,1.0,2.1,4.7,7.0,3.8,209.5,1.0,3.8,75.1,209.5
bupa,76.7,59.7,1.0,1.8,1.0,1.0,1.4,1.7,1.3,81.6,1.0,1.3,38.4,81.6
cleveland,102.2,102.8,30.8,128.5,1.6,27.2,34.5,128.5,33.4,171.2,2.1,28.2,96.2,167.3
ecoli,56.8,62.2,2.9,82.3,1.0,2.9,41.8,82.3,2.9,96.5,1.0,2.9,52.7,96.5
glass,51.1,60.0,103.2,103.2,1.0,3.1,56.0,103.2,3.7,117.1,1.0,3.7,57.0,117.1
heart,44.7,37.9,38.8,38.8,1.5,13.5,6.8,38.6,24.6,99.0,1.1,19.3,33.7,92.0
ionosphere,19.3,26.8,54.9,54.9,1.0,2.0,18.5,54.3,2.9,84.8,1.2,2.9,22.8,84.8
pima,125.0,95.5,7.3,7.3,1.0,1.0,4.5,7.2,1.9,171.8,1.0,1.9,74.6,171.8
sonar,17.8,28.0,1.0,122.4,1.0,1.0,32.1,122.4,2.0,160.5,1.0,2.0,28.0,160.5


In [148]:
if save:    
    bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_51409/1482801916.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [144]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.drop(['glass', 'yeast'], inplace=True)
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)
table_attribute

,incl 1e-4,msecvx-i50-reducts,relabel-maecvx-reducts-default,relabel-maecvx-c50-reducts,relabel-maecvx-i50-reducts,relabel-maecvx-i95-reducts,relabel-maecvx-i50+c50-reducts,relabel-maecvx-i95+c50-reducts,relabel-msecvx-reducts-default,relabel-msecvx-c50-reducts,relabel-msecvx-i50-reducts,relabel-msecvx-i95-reducts,relabel-msecvx-i50+c50-reducts,relabel-msecvx-i95+c50-reducts
australian,5.154734,7.138659,11.121858,11.121858,1.4,5.913333,2.888766,10.56365,11.362474,13.355079,2.0,7.297619,4.077074,12.911711
bands,6.292805,5.270798,11.560833,11.560833,1.1,3.4,2.215,8.22381,13.75,18.905128,3.8,13.75,3.94959,18.905128
bupa,4.202797,4.312907,1.8,1.258333,0.3,1.8,0.583333,1.341667,6.0,6.0,3.0,6.0,2.724225,6.0
cleveland,5.449101,4.82389,10.597313,12.108793,1.366667,10.171746,2.26957,11.986779,11.378068,12.529917,2.966667,10.80406,4.495994,12.466548
ecoli,3.854646,3.662146,5.2,6.934423,3.8,5.2,2.867183,6.923828,5.2,6.943765,3.6,5.2,2.916294,6.943765
heart,4.904402,4.227588,8.869999,8.869999,1.3,6.601625,1.368769,8.417814,9.876584,11.797238,1.95,9.0648,3.576542,11.463226
ionosphere,4.527456,8.510326,32.370461,32.370461,6.5,17.0,4.499291,29.609007,21.8,32.622067,3.85,21.8,3.611969,32.622067
pima,5.080997,4.702172,7.201786,7.201786,1.9,8.0,1.62,6.516786,8.0,8.0,2.6,8.0,3.263146,7.99645
sonar,7.798125,5.855025,60.0,60.0,4.7,60.0,5.175192,59.344769,60.0,60.0,3.6,60.0,5.855025,60.0
spectfheart,8.24058,31.377692,0.0,0.0,0.0,0.0,0.0,0.0,39.6,33.223854,0.9,39.6,1.25,31.070175


In [149]:
if save:
    bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_51409/357892810.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [170]:
from scipy import stats
import scikit_posthocs as sph

In [185]:
subject = table_acc

In [186]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('max', axis='columns', inplace=True)

In [187]:
no_mean

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
australian,0.824936,0.787517,0.813182,0.82039,0.812028
bands,0.664654,0.675141,0.676793,0.60893,0.658848
bupa,0.593125,0.600724,0.594187,0.640177,0.589394
cleveland,0.317538,0.290397,0.314681,0.283667,0.293392
ecoli,0.700635,0.720659,0.69504,0.681103,0.682324
glass,0.668254,0.679067,0.641468,0.497123,0.668552
heart,0.730833,0.7675,0.734167,0.75,0.769167
ionosphere,0.910721,0.912468,0.91078,0.889655,0.901234
pima,0.680919,0.687633,0.673859,0.660827,0.657748
sonar,0.776288,0.786212,0.721313,0.771187,0.720051


In [208]:
stats.wilcoxon(no_mean['cv-thres'],no_mean['incl 1e-6'], alternative='two-sided')

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


WilcoxonResult(statistic=19.0, pvalue=0.03546470751403722)

In [128]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=72.60000000000002, pvalue=1.5315170910874449e-09)

In [129]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
incl 1e-6,1.000000,1.0,1.0,0.757743,0.858925
incl 1e-4,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-3,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-2,0.757743,1.0,1.0,1.000000,1.000000
cv-thres,0.858925,1.0,1.0,1.000000,1.000000


In [65]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

incl 1e-3     0.04599
incl 1e-6    0.047052
cv-thres     0.050783
incl 1e-4    0.050873
incl 1e-2    0.181944
dtype: object

In [189]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
incl 1e-6 &  2.433333 \\
incl 1e-4 &  2.600000 \\
incl 1e-3 &  2.900000 \\
cv-thres  &  3.433333 \\
incl 1e-2 &  3.633333 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [67]:
ranks['lower-new-check'].value_counts()

KeyError: 'lower-new-check'

In [68]:
ranks.apply(pd.value_counts)

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
1.0,3,5.0,1.0,3,2
2.0,4,3.0,3.0,1,2
2.5,1,NaN,1.0,1,1
3.0,5,1.0,7.0,1,2
4.0,1,1.0,3.0,3,5
5.0,1,5.0,NaN,6,3


In [188]:
median = no_mean.median(axis='rows')
median

incl 1e-6    0.680919
incl 1e-4    0.687633
incl 1e-3    0.676793
incl 1e-2    0.660827
cv-thres     0.668552
dtype: float64

On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000


0.03